[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Broadcasting


## What you will be able to do

Combine arrays of different shapes, such as every reading and its own station's mean, predict the
shape of the result before running anything, and read the error NumPy gives when two shapes do not
fit.

## The idea

### The problem

Each station has its own mean for the year, 8.0 degrees at Bergen and -4.48 at Svalbard. A question
such as "how far above its usual was each station at noon on 1 July" needs every reading minus its
own station's mean: Bergen's mean taken from all 8,760 of Bergen's readings, Oslo's from Oslo's, and
so on. The readings have shape (4, 8760) and the means have shape (4,), so the two arrays are not the
same shape.

One answer is a second array of shape (4, 8760) that repeats each mean 8,760 times, and a
subtraction of that. It works, and it fills 280,320 bytes with copies of four numbers. Another is a
loop over the stations, which is what arrays were meant to replace. NumPy stretches the smaller array
to fit the larger one instead, without copying anything, whenever the two shapes fit by a rule.

### What broadcasting is

> In NumPy, **broadcasting** is the rule for combining arrays of different shapes value by value.
> Compare the two shapes from the last dimension backward. Two dimensions fit when they are equal or
> when one of them is 1, and a shape with fewer dimensions counts as having 1s on its left. A
> dimension of length 1 is stretched to the other's length without copying any values.

### Why it works that way

The rule is easiest to read with the shapes written one above the other, lined up on the right:

```
readings   (4, 8760)
means         (4,)
```

The last dimensions are 8760 and 4, which are neither equal nor 1, so the subtraction fails. Give the
means a second dimension of length 1, shape (4, 1), and the comparison becomes 8760 against 1, which
fits, then 4 against 4, which fits too. The column of four means is stretched along the hours, so
every reading at a station meets that station's mean.

Because the shapes are compared from the right, the one dimension of a row of 8,760 values, shape
(8760,), lines up with the hours, and the row fits the readings with no help. The rule looks only at
the lengths of the dimensions, never at what they stand for. A table of four stations by four
quarters of the year, minus the four station means, fits as well, and NumPy raises nothing, though
the means meet the quarters instead of the stations.

Stretching costs nothing, because NumPy reads the same value again for every place it stands for. A
column of four means broadcast across 8,760 hours uses the memory of four numbers, and only the
result of the arithmetic is a new array of the full shape.

### Where this shows up

Converting units, `readings * 9 / 5 + 32`, is the simplest broadcast, one number stretched over every
value. Standardizing the columns of a table, subtracting each column's mean and dividing by its
standard deviation, broadcasts a row of means down every row, which is what scaling does in the
**scikit-learn, Deep Dive** guide. A color image of shape (height, width, 3), scaled by three channel
weights, broadcasts an array of shape (3,). The `np.ix_` grid in the **Fancy Indexing** notebook was
a (2, 1) column broadcast against a (1, 12) row.

### What this notebook covers

- a single number with an array
- the rule, checked with `np.broadcast_shapes`
- a row of hours against every station, and a column of stations against every hour
- `np.newaxis` and `keepdims=True`, to make the column
- three dimensions, with each station's daily cycle against each of its days
- a table of counts from a column of thresholds
- `np.broadcast_to`, to see that broadcasting copies nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

grid = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])
print(grid - np.array([1.0, 2.0, 3.0]))
print(grid - np.array([[1.0], [4.0]]))
```

```
[[0. 0. 0.]
 [3. 3. 3.]]
[[0. 1. 2.]
 [0. 1. 2.]]
```

A row of three values was subtracted from both rows of the grid, and a column of two values from all
three columns.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### A number with every value

In [2]:
fahrenheit = readings * 9 / 5 + 32

print(readings.shape, fahrenheit.shape)
print(readings[0, :3])
print(fahrenheit[0, :3])

(4, 8760) (4, 8760)
[-3.6 -3.7 -3.7]
[25.52 25.34 25.34]


A single number has no dimensions, so it fits any shape, and NumPy uses it for every value. The
**Why NumPy** notebook converted the readings this way before broadcasting had a name.

### Checking the rule with np.broadcast_shapes

`np.broadcast_shapes` applies the rule to shapes alone, without making an array, and returns the
shape the result would have:

In [3]:
pairs = [
    ((4, 8760), (8760,)),
    ((4, 8760), (4, 1)),
    ((4, 365, 24), (4, 1, 24)),
    ((3, 1, 1), (4, 8760)),
]
for first, second in pairs:
    print(first, "with", second, "gives", np.broadcast_shapes(first, second))

(4, 8760) with (8760,) gives (4, 8760)
(4, 8760) with (4, 1) gives (4, 8760)
(4, 365, 24) with (4, 1, 24) gives (4, 365, 24)
(3, 1, 1) with (4, 8760) gives (3, 4, 8760)


In the last pair, (4, 8760) counts as (1, 4, 8760), and every 1 stretches, so the result has three
dimensions, (3, 4, 8760). When two shapes do not fit, `np.broadcast_shapes` raises and names the
pair:

In [4]:
try:
    np.broadcast_shapes((4, 8760), (4,))
except ValueError as error:
    print(error)

shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (4, 8760) and arg 1 with shape (4,).


### A row against every station

`readings[0]` has shape (8760,), which lines up with the hours, so subtracting it takes Bergen's
reading from every station's at the same hour:

In [5]:
from_bergen = readings - readings[0]

print(from_bergen.shape)
print(np.nanmean(from_bergen, axis=1).round(2))

(4, 8760)
[  0.   -1.5 -12.5  -4.5]


On average Oslo was 1.5 degrees colder than Bergen at the same hour, Tromso 4.5 and Svalbard 12.5,
and Bergen's own row gives 0.

### A column against every hour

The station means have shape (4,), which would line up with the hours. `np.newaxis` in an index adds
a dimension of length 1, so `means[:, np.newaxis]` is a column of shape (4, 1), which lines up with
the stations:

In [6]:
means = np.nanmean(readings, axis=1)
column = means[:, np.newaxis]
anomaly = readings - column

print(means.shape, column.shape)
print(means.round(2))
print(readings[:, hour_of_year(7, 1, 12)])
print(anomaly[:, hour_of_year(7, 1, 12)].round(2))
print(np.nanmax(anomaly, axis=1).round(2))

(4,) (4, 1)
[ 8.    6.5  -4.48  3.5 ]
[19.1 17.5  6.4 14.3]
[11.1  11.   10.88 10.8 ]
[12.8  12.8  12.78 12.7 ]


`anomaly` is every reading minus its own station's mean. At noon on 1 July every station was about 11
degrees above its yearly mean, though the readings themselves ran from 6.4 at Svalbard to 19.1 at
Bergen. No station rose more than 12.8 degrees above its mean, which is the most the formula in Setup
can add, with 9 degrees of season, 3 of day and 0.8 of wobble.

### keepdims=True keeps the column

`keepdims=True` leaves the averaged dimension in place with length 1, so the mean comes out as a
column, ready to subtract:

In [7]:
kept = np.nanmean(readings, axis=1, keepdims=True)

print(kept.shape)
print(np.array_equal(readings - kept, anomaly, equal_nan=True))

(4, 1)
True


The two ways give the same anomalies. `equal_nan=True` counts Svalbard's missing hours as equal,
since `nan` fails every comparison, its own included.

### Broadcasting in three dimensions

Reshaped to (4, 365, 24), the readings have a dimension for the hour of the day. Their mean over the
days, with `keepdims=True`, has shape (4, 1, 24), which is each station's mean at each hour of the
day, its daily cycle:

In [8]:
by_day = readings.reshape(4, 365, 24)
cycle = np.nanmean(by_day, axis=1, keepdims=True)
without_cycle = by_day - cycle

print(cycle.shape, without_cycle.shape)
print(cycle[:, 0, 3].round(2))
print(cycle[:, 0, 15].round(2))

(4, 1, 24) (4, 365, 24)
[ 5.    3.5  -7.48  0.5 ]
[11.    9.5  -1.48  6.5 ]


Every station was 3 degrees below its yearly mean at 03:00 and 3 degrees above it at 15:00, which is
the daily swing in the formula. The cycle's 1 stretched along the 365 days, and the rule handled three
dimensions in the same way as two.

### A table from a column of thresholds

Three thresholds shaped (3, 1, 1), against the readings' (4, 8760), give (3, 4, 8760), every
threshold compared with every reading. Summing over the hours leaves a table of counts:

In [9]:
thresholds = np.array([5, 10, 15])
above = readings > thresholds[:, np.newaxis, np.newaxis]
table = above.sum(axis=2)

print(above.shape, above.size)
for threshold, counts in zip(thresholds, table):
    print(f"above {threshold:2}", counts)

(3, 4, 8760) 105120
above  5 [5352 4844  711 3891]
above 10 [3718 3197    0 1847]
above 15 [1621 1043    0  140]


Svalbard was above 5 degrees for 711 hours and never above 10. The comparison made one answer for
every threshold, station and hour, 105,120 in all, so a long list of thresholds makes a large array
in the middle of the calculation, even though the table at the end is small.

### Broadcasting copies nothing

`np.tile` builds the repeated array that the problem at the top of the notebook described.
`np.broadcast_to` returns what broadcasting uses instead:

In [10]:
tiled = np.tile(column, (1, HOURS))
stretched = np.broadcast_to(column, (4, HOURS))

print(tiled.shape, tiled.nbytes)
print(stretched.shape, np.shares_memory(stretched, column))
print(column.nbytes, stretched.flags.writeable)

(4, 8760) 280320
(4, 8760) True
32 False


Both have shape (4, 8760). The tiled array holds 280,320 bytes of copies, and the stretched one shares
the 32 bytes of the column, reading each mean again for every hour. NumPy marks it as read-only,
because one stored value stands in for 8,760 places.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/08-broadcasting-solutions.ipynb).

**1.** Subtract Oslo's row, row 1, from every station, and print each station's mean difference from Oslo with `np.nanmean`, rounded to two places.

In [11]:
# your code here

**2.** Use `np.broadcast_shapes` to print the shape that (365, 24) with (24,) gives, and the shape that (365, 24) with (365, 1) gives.

In [12]:
# your code here

**3.** Compute each station's anomaly from its own mean with `keepdims=True`, and print the anomalies at noon on 1 January, rounded to two places.

In [13]:
# your code here

**4.** Count the hours below -10, -5 and 0 degrees at every station in one comparison, using a column of thresholds, and print the table.

In [14]:
# your code here

**5.** Reshape Bergen's row to (365, 24), subtract the mean of each hour of the day across the year, and print the result's shape and its largest value, rounded to two places.

In [15]:
# your code here

## Common errors

### ValueError: operands could not be broadcast together with shapes (4,8760) (4,)

In [16]:
readings - means

ValueError: operands could not be broadcast together with shapes (4,8760) (4,) 

The message lists the shapes in the order the operands were written, without spaces. Lined up on the
right, they compare 8760 with 4:

```
readings   (4, 8760)
means         (4,)
```

Neither 8760 nor 4 is 1, so the subtraction fails. Make the means a column, `means[:, np.newaxis]` or
`np.nanmean(readings, axis=1, keepdims=True)`, so that the 4 lines up with the stations.

### ValueError: could not broadcast input array from shape (4,) into shape (4,24)

In [17]:
scratch = readings.copy()
scratch[:, :24] = means

ValueError: could not broadcast input array from shape (4,) into shape (4,24)

Assignment follows the same rule, with the values on the right stretched to fit the place on the
left, here the first day at every station, shape (4, 24). The four means line up with the 24 hours,
and do not fit. A column fits:

In [18]:
scratch[:, :24] = column

print(scratch[:, :2].round(2))

[[ 8.    8.  ]
 [ 6.5   6.5 ]
 [-4.48 -4.48]
 [ 3.5   3.5 ]]


### No error, and the means subtracted along the wrong dimension

`quarters` holds each station's mean for each quarter of the year, 2,190 hours each, so it is a (4, 4)
table of stations by quarters. Subtracting the four yearly means should give each quarter's distance
from its station's mean:

In [19]:
quarters = np.nanmean(readings.reshape(4, 4, 2190), axis=2)

print((quarters - means).round(2))

[[ -7.13   5.37  19.61   0.63]
 [ -8.63   3.87  18.11  -0.87]
 [-19.63  -7.13   7.11 -11.87]
 [-11.63   0.87  15.11  -3.87]]


Bergen's third quarter, at 19.61 degrees above its mean, cannot be right for a station that never rose
12.8 degrees above it in any hour. A (4, 4) table and four means fit by the rule, which lines the means
up with the last dimension, the quarters, so Svalbard's mean was taken from every station's third
quarter. A square array is where this happens without an error, because both of its dimensions have
the same length. Say which dimension the means belong to:

In [20]:
print((quarters - column).round(2))

[[-7.13  3.87  7.13 -3.87]
 [-7.13  3.87  7.13 -3.87]
 [-7.15  3.85  7.11 -3.89]
 [-7.13  3.87  7.13 -3.87]]


Every station was about 7 degrees below its mean in the first quarter and 7 above it in the third.

### No error, and a (4, 4) table where four values were meant

`kept` is a column, shape (4, 1), and the readings at noon on 1 July are a row of four, shape (4,):

In [21]:
noon_readings = readings[:, hour_of_year(7, 1, 12)]
difference = noon_readings - kept

print(noon_readings.shape, kept.shape, difference.shape)

(4,) (4, 1) (4, 4)


A column against a row is the threshold table again, every station's noon reading minus every
station's mean, sixteen values of which only the four on the diagonal answer the question. Keep both
sides the same kind, two rows or two columns:

In [22]:
print((noon_readings - means).round(2))

[11.1  11.   10.88 10.8 ]


## Recap

- Combine arrays of different shapes when, compared from the right, each pair of dimensions is equal
  or 1
- Check two shapes with `np.broadcast_shapes` before making anything
- Turn four values into a column with `[:, np.newaxis]` or `keepdims=True`
- Read the shapes in a broadcast error lined up on the right
- Check square arrays and a column against a row, which broadcast without an error

## What is next

The **Vectorized Math** notebook works through the functions that apply to every value at once,
`np.sqrt`, `np.clip` and `np.where` among them, and the warnings they give in place of errors.

---

&#8592; **Previous:** [Fancy Indexing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/07-fancy-indexing.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Vectorized Math](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/09-vectorized-math.ipynb) &#8594;
